# Laboratorio 6: Bases de Datos con Python — SQLite y MongoDB

Bienvenido al Laboratorio 6. En esta sesión exploraremos las dos caras de la moneda del almacenamiento de datos: el mundo **Relacional (SQL)** con SQLite y el mundo **No Relacional (NoSQL)** con MongoDB. Aprenderás a modelar datos, garantizar la integridad referencial y manipular documentos de forma eficiente desde Python.

## 📋 Tabla de Contenidos

1. [PARTE 1: SQLite desde la Terminal](#-sqlite-terminal)
2. [PARTE 2: Modelo Relacional (User-Post-Comment)](#-modelo-relacional)
3. [PARTE 3: SQLModel en Python](#-sqlmodel)
4. [PARTE 4: Operaciones CRUD con SQLModel](#-crud-sql)
5. [PARTE 5: MongoDB desde Mongo Shell](#-mongo-shell)
6. [PARTE 6: PyMongo en Python](#-pymongo)
7. [PARTE 7: MongoEngine (ODM)](#-mongoengine)
8. [PARTE 8: Comparativa: SQL vs NoSQL](#-comparativa)

---

En una base de datos relacional, los datos se organizan en tablas conectadas por relaciones. Usaremos el modelo **User -> Post -> Comment** para entender la **Integridad Referencial**.

| Tabla | Campos Principales | Relación |
| :--- | :--- | :--- |
| **User** | `id (PK)`, `username`, `married (bool)`, `dates` | Dueño de los Posts. |
| **Post** | `id (PK)`, `title`, `content`, `user_id (FK)` | Pertenece a 1 Usuario (1:M). |
| **Comment**| `id (PK)`, `content`, `post_id (FK)`, `user_id (FK)` | Conecta Usuario y Post. |

---

<a id='-sqlite-terminal'></a>
## 🔹 1: PARTE 1: SQLite desde la Terminal

SQLite es un motor de base de datos ligero que **no requiere un servidor**. Todo se almacena en un simple archivo local. Antes de usar Python, es vital entender cómo funciona desde la línea de comandos.

### Conceptos Clave:
- **Crear DB:** `sqlite3 mi_base.db` en la terminal.
- **Ver Esquema:** `.schema` muestra el código SQL de tus tablas.
- **Comandos Básicos:**
  - `CREATE TABLE`: Define la estructura.
  - `INSERT`: Agrega datos.
  - `SELECT`: Consulta datos.

```sql
-- 1. Definición del esquema (Tablas y Relaciones)
CREATE TABLE IF NOT EXISTS user (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT NOT NULL UNIQUE,
    married BOOLEAN DEFAULT 0,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS post (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    content TEXT NOT NULL,
    user_id INTEGER NOT NULL,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id) REFERENCES user (id)
);

CREATE TABLE IF NOT EXISTS comment (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    content TEXT NOT NULL,
    user_id INTEGER NOT NULL,
    post_id INTEGER NOT NULL,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id) REFERENCES user (id),
    FOREIGN KEY (post_id) REFERENCES post (id)
);

-- 2. Carga de datos iniciales (Seed Data)
INSERT INTO user (username, first_name, last_name, married) 
VALUES ('jorge_laco', 0);

INSERT INTO post (title, content, user_id) 
VALUES ('Mi primer post', 'Contenido sobre Cloud Architecture', 1);

INSERT INTO comment (content, user_id, post_id) 
VALUES ('Excelente post!', 1, 1);

-- 3. Consultar los datos (Verificación)
SELECT * FROM user;
SELECT * FROM post;
SELECT * FROM comment;

-- 4. Mostrar la estructura de las tablas
PRAGMA table_info(user);
PRAGMA table_info(post);
PRAGMA table_info(comment);
```

---

<a id='-sqlmodel'></a>
## 🔹 3: PARTE 2: SQLModel en Python

**SQLModel** es la librería moderna de Python que combina el poder de SQLAlchemy con la simplicidad de Pydantic. Es la forma más limpia de manejar SQL hoy en día.

In [ ]:
!pip install sqlmodel

In [ ]:
from datetime import datetime
from typing import List, Optional
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select, text

# 1. Definición de Modelos
class User(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    username: str = Field(index=True, unique=True)
    first_name: str
    last_name: str
    married: bool = False
    created_at: datetime = Field(default_factory=datetime.now)
    
    posts: List["Post"] = Relationship(back_populates="author")

class Post(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str
    content: str
    user_id: int = Field(foreign_key="user.id")
    created_at: datetime = Field(default_factory=datetime.now)
    
    author: User = Relationship(back_populates="posts")
    comments: List["Comment"] = Relationship(back_populates="post")

class Comment(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    content: str
    user_id: int = Field(foreign_key="user.id")
    post_id: int = Field(foreign_key="post.id")
    created_at: datetime = Field(default_factory=datetime.now)
    
    post: Post = Relationship(back_populates="comments")

# 2. Configuración del Engine (SQLite Local)
sqlite_file = "blog.db"
sqlite_url = f"sqlite:///{sqlite_file}"
engine = create_engine(sqlite_url, echo=False)

def create_db_and_tables():
    SQLModel.metadata.create_all(engine)
    print("✅ Base de datos y tablas creadas exitosamente.")

---

<a id='-crud-sql'></a>
## 🔹 4: PARTE 4: Operaciones CRUD Completas

Aprenderemos a crear, leer, actualizar y eliminar registros usando funciones limpias.

In [ ]:
def create_sample_data():
    with Session(engine) as session:
        # CREATE
        u1 = User(username="jdoe", first_name="John", last_name="Doe", married=True)
        p1 = Post(title="Mi primer post", content="Hola mundo SQLModel", author=u1)
        c1 = Comment(content="¡Excelente post!", post=p1, user_id=1)
        
        session.add(u1)
        session.add(p1)
        session.add(c1)
        session.commit()
        print("📝 Datos de prueba insertados.")

def read_and_update():
    with Session(engine) as session:
        # READ (select)
        statement = select(User).where(User.username == "jdoe")
        user = session.exec(statement).first()
        print(f"🔍 Usuario encontrado: {user.first_name} {user.last_name}")
        
        # UPDATE
        user.married = False
        session.add(user)
        session.commit()
        print("🔄 Estado de matrimonio actualizado.")

def delete_user(username: str):
    with Session(engine) as session:
        user = session.exec(select(User).where(User.username == username)).first()
        if user:
            session.delete(user)
            session.commit()
            print(f"🗑️ Usuario {username} eliminado.")

# Ejecutamos todo el flujo
create_db_and_tables()
create_sample_data()
read_and_update()
raw_sql_query()

In [ ]:
def raw_sql_query(table="user"):
    with Session(engine) as session:
        # Al agregar .mappings(), cada fila es un diccionario: {"id": 1, "username": "jorge"}
        result = session.exec(text(f"SELECT * FROM {table}")).mappings()
        
        for row in result:
            print(f"💾 Registro en {table}: {row}")

---

<a id='-mongo-shell'></a>
## 🔹 5: PARTE 5: MongoDB desde Mongo Shell

A diferencia de SQL, MongoDB es **NoSQL** y está **orientado a documentos (BSON)**. No tiene un esquema rígido (schema-less), lo que permite guardar objetos complejos con diferentes estructuras en la misma colección.

### Comandos de Mongo Shell:
- `use blog_db`: Crea o cambia de base de datos.
- `db.users.insertOne({name: 'Ana'})`: Crea colección y documento.
- `db.users.find({age: {$gt: 18}})`: Filtra documentos.
- `db.users.updateOne({name: 'Ana'}, {$set: {married: true}})`: Actualiza.
- **BSON:** Es el formato binario de JSON que usa Mongo para ser más rápido y soportar más tipos de datos.

---

<a id='-pymongo'></a>
## 🔹 6: PARTE 6: PyMongo en Python

PyMongo es el driver oficial de MongoDB. Es ideal para cuando necesitas control total sobre los diccionarios.

In [ ]:
!pip install pymongo

In [ ]:
import pymongo

# Conexión (Asumiendo Mongo local)
try:
    client = pymongo.MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=2000)
    db = client["lab_database"]
    users_col = db["users"]

    # INSERT
    user_doc = {"username": "marta99", "first_name": "Marta", "tags": ["python", "mongo"]}
    result = users_col.insert_one(user_doc)
    
    # FIND
    found = users_col.find_one({"username": "marta99"})
    print(f"🍃 Documento en PyMongo: {found}")
except Exception as e:
    print("⚠️ MongoDB no disponible localmente. Mostrando lógica de código solamente.")

---

<a id='-mongoengine'></a>
## 🔹 7: PARTE 7: MongoEngine (ODM)

**MongoEngine** es un ODM (Object-Document Mapper). Nos permite definir clases para nuestros documentos de forma similar a como lo hicimos en SQLModel.

In [ ]:
!pip install mongoengine

In [ ]:
from mongoengine import Document, StringField, BooleanField, ReferenceField, ListField, connect

connect(db="test_db", host="localhost", port=27017)

# 1. Definición de Modelos
class MUser(Document):
    username = StringField(required=True, unique=True)
    first_name = StringField()
    last_name = StringField()
    married = BooleanField(default=False)

class MPost(Document):
    title = StringField(required=True)
    content = StringField()
    author = ReferenceField(MUser, reverse_delete_rule=2) # 2 = CASCADE

# 2. CRUD Operations
def create_mongo_data():
    try:
        # CREATE
        u1 = MUser(username="jdoe", first_name="John", last_name="Doe", married=True).save()
        p1 = MPost(title="Mi primer post NoSQL", content="Hola desde MongoEngine", author=u1).save()
        print("📝 Documentos de prueba insertados en MongoDB.")
    except NotUniqueError:
        print("⚠️ El usuario ya existe.")

def read_and_update_mongo():
    # READ (objects)
    user = MUser.objects(username="jdoe").first()
    if user:
        print(f"🔍 Documento encontrado: {user.first_name} {user.last_name}")
        
        # UPDATE
        user.married = False
        user.save()
        print("🔄 Documento de Mongo actualizado.")

def delete_mongo_user(username: str):
    # DELETE
    user = MUser.objects(username=username).first()
    if user:
        user.delete()
        print(f"🗑️ Documento {username} eliminado de la colección.")

# 3. Ejecución del Flujo
print("💡 Con MongoEngine, trabajamos con Documentos en lugar de Tablas.")
create_mongo_data()
read_and_update_mongo()
# delete_mongo_user("jdoe")

---

<a id='-comparativa'></a>
## 🔹 8: Comparativa: SQL vs NoSQL

Para cerrar, analicemos cuándo usar cada una:

| Característica | SQLite (Relacional) | MongoDB (Documental) |
| :--- | :--- | :--- |
| **Esquema** | Rígido (Tablas fijas) | Flexible (Sin esquema) |
| **Relaciones** | JOINs potentes | Referencias o Embebido |
| **Integridad** | Alta (Constraints/FK) | Responsabilidad del programador |
| **Escalabilidad**| Vertical (Mejor CPU/RAM) | Horizontal (Más servidores) |
| **Caso de Uso** | Finanzas, ERPs, Apps Locales | Redes Sociales, Big Data, CMS |

### Conclusión
No existe una "mejor" base de datos, sino la herramienta adecuada para el problema adecuado. **SQLite** es imbatible para aplicaciones locales y datos estructurados, mientras que **MongoDB** brilla cuando la rapidez de desarrollo y la flexibilidad de los datos son la prioridad.